# WAV1 mechanism factorization — seed42 sequential Kaggle
Menjalankan `HP1 → WAV_L1 → WAV_L2 → WAV_RAWFUSE` satu per satu. Ini **mechanistic screen**, bukan pencarian model terbaik. `WAV1_REF` tidak diretrain. Locked test tetap tertutup.
Input wajib: private core `faruq-v3-experiment-core-v1` dan add-on `faruq-v3-wav1-factorization-addon-v1`.


In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
core=sorted(INPUT.rglob('af2_spectral_kaggle_manifest.json'))
addon=sorted(INPUT.rglob('wav1_factorization_kaggle_manifest.json'))
if len(core)!=1 or len(addon)!=1: raise FileNotFoundError(f'STOP CEPAT: core={core}, addon={addon}')
print('FAST INPUT PREFLIGHT PASS'); print('CORE:',core[0]); print('ADDON:',addon[0])


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile,torch
os.chdir(WORK); REPO=WORK/'coffee-bean-detection'; BRANCH='agent/wav1-mechanism-factorization'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip(); print('COMMIT:',COMMIT)
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_wav1_factorization.py'],cwd=REPO,check=True)
from coffee_detector.experiments.prepare_wav1_factorization_kaggle import prepare_wav1_factorization_kaggle_input,restore_wav1_factorization_kaggle_run
from coffee_detector.wav1_factorization.audit import run_static_audit
from coffee_detector.wav1_factorization import TRAIN_ARMS,WAV1FactorizationEnhancer,frozen_arm_config
DATA,ARTIFACTS,INPUT_CONTRACT=prepare_wav1_factorization_kaggle_input(INPUT,WORK)
assert INPUT_CONTRACT['decision']=='PASS' and INPUT_CONTRACT['test_images_accessed'] is False
D0=ARTIFACTS['D0_seed42_best.pt']; D0FT_REF=ARTIFACTS['D0FT_seed42_reference.json']; WAV1_REF=ARTIFACTS['WAV1_seed42_result.json']
print('FULL INPUT CONTRACT PASS'); print('DATA:',DATA); print('D0:',D0); print('D0FT REF:',D0FT_REF); print('WAV1 REF:',WAV1_REF)
OUT=WORK/'wav1-mechanism-factorization-v1'; OUT.mkdir(exist_ok=True); STATIC=OUT/'static_audit.json'
audit=run_static_audit(D0,STATIC); print('STATIC:',audit['decision']); print('WAV1 BITWISE REF:',audit['wav1_ref_bitwise_equal_to_confirmed_operator'])
if audit['decision']!='PASS': raise RuntimeError(f'STOP: static audit gagal: {audit}')
previous_det=torch.are_deterministic_algorithms_enabled()
try:
    torch.use_deterministic_algorithms(True,warn_only=False)
    for arm in TRAIN_ARMS:
        probe=torch.rand(1,3,65,63,device='cuda:0',requires_grad=True)
        frontend=WAV1FactorizationEnhancer(frozen_arm_config(arm)).to('cuda:0')
        first=frontend(probe); second=frontend(probe.detach())
        if not torch.equal(first.detach(),second.detach()): raise RuntimeError(f'{arm} tidak bitwise repeatable pada deterministic CUDA smoke')
        first.mean().backward(); assert probe.grad is not None and torch.isfinite(probe.grad).all()
        print(f'DETERMINISTIC CUDA SMOKE {arm}: PASS')
finally:
    torch.use_deterministic_algorithms(previous_det)
RESUME_ROOT=WORK/'wav1-factorization-resume-input'
if RESUME_ROOT.exists(): shutil.rmtree(RESUME_ROOT)
RESUME_ROOT.mkdir(parents=True)
resume_zips=sorted({*INPUT.rglob('wav1-factorization-stage1-output.zip'),*INPUT.rglob('HP1_seed42_output.zip'),*INPUT.rglob('WAV_L1_seed42_output.zip'),*INPUT.rglob('WAV_L2_seed42_output.zip'),*INPUT.rglob('WAV_RAWFUSE_seed42_output.zip')})
for i,archive in enumerate(resume_zips):
    dst=RESUME_ROOT/f'zip_{i:02d}_{archive.stem}'; dst.mkdir(parents=True)
    with zipfile.ZipFile(archive,'r') as h: h.extractall(dst)
    print('RESUME ZIP EXTRACTED:',archive,'->',dst)
print('RESUME ZIP COUNT:',len(resume_zips))


In [ ]:
ARMS=('HP1','WAV_L1','WAV_L2','WAV_RAWFUSE'); SNAPSHOT_BASE=WORK/'wav1-factorization-stage1-output'
def snapshot_stage1():
    a=shutil.make_archive(str(SNAPSHOT_BASE),'zip',OUT); print('SNAPSHOT READY:',a,flush=True); return Path(a)
def restore_exact_arm(arm,config):
    r=restore_wav1_factorization_kaggle_run(INPUT,OUT,arm=arm,seed=42,d0_checkpoint=D0,config=config)
    if r is None and any(RESUME_ROOT.iterdir()): r=restore_wav1_factorization_kaggle_run(RESUME_ROOT,OUT,arm=arm,seed=42,d0_checkpoint=D0,config=config)
    return r
def run_arm(arm):
    config=REPO/f'configs/wav1_factorization/{arm}_yolo26n.yaml'; restored=restore_exact_arm(arm,config)
    result=OUT/'val_reports'/f'{arm}_seed42_result.json'; log=OUT/f'{arm}_seed42_run.log'
    if result.is_file(): print(f'REUSE COMPLETE {arm}: {result}',flush=True); snapshot_stage1(); return
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_wav1_factorization_arm','--arm',arm,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--d0-checkpoint',str(D0),'--static-audit',str(STATIC),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
    print(f'START {arm} | restored={restored}',flush=True)
    with log.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    previous=None
    while process.poll() is None:
        csv=OUT/arm/f'{arm}_seed42'/'results.csv'; epoch=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epoch!=previous: print(f'{arm}: {epoch}/50 epoch | log={log}',flush=True); previous=epoch
        time.sleep(120)
    if process.returncode:
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-180:]) if log.is_file() else '<log tidak ditemukan>'
        raise RuntimeError(f'{arm} gagal: returncode={process.returncode}\n--- {log.name} tail ---\n{tail}')
    if not result.is_file():
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-180:]) if log.is_file() else '<log tidak ditemukan>'
        raise RuntimeError(f'Hasil {arm} tidak ditemukan: {result}\n--- {log.name} tail ---\n{tail}')
    payload=json.loads(result.read_text(encoding='utf-8')); assert payload['evaluation_split']=='val' and payload['test_images_accessed'] is False
    print(f'SELESAI {arm}',flush=True); snapshot_stage1()
for arm in ARMS: run_arm(arm)
print('SCREEN COMPLETE')


In [ ]:
from coffee_detector.experiments.run_faruq_v3_wav1_factorization_decision import run_factorization_report
arm_results=[OUT/'val_reports'/f'{arm}_seed42_result.json' for arm in ARMS]
REPORT=OUT/'val_reports'/'wav1_factorization_seed42_report.json'
report=run_factorization_report(D0FT_REF,WAV1_REF,arm_results,REPORT)
print('REFERENCE WAV1 GAIN:',report['references']['wav1_gain_vs_d0ft'])
for row in report['arms']:
    print('\n',row['arm']); print('metrics=',row['metrics']); print('gain=',row['gain_vs_d0ft']); print('preservation=',row['wav1_gain_preservation']); print('class_corr=',row['per_class_pattern']['pearson_delta_correlation'])
archive=snapshot_stage1(); print('FINAL ZIP:',archive); print('REPORT:',REPORT)
print('STOP HERE. Jangan buka seed 123/2026 atau locked test dari notebook ini.')
